In [ ]:
# =============================================
# Step 1: Mount Google Drive
# =============================================
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
cd /content/drive/MyDrive/

/content/drive/MyDrive


In [ ]:
# ============================================================
# SINGLE UNIFIED PIPELINE
#
# What this does:
# 1) Load data + embeddings
# 2) Build ground truth
# 3) Create ONE fixed train/test split
# 4) Save split metadata
# 5) For the SAME fixed test set, generate masks for:
#       - MCAR
#       - MAR
#       - MNAR
#    and for each scenario generate masks at 5 missing ratios:
#       0.1, 0.2, 0.3, 0.4, 0.5
#    with 10 masks per ratio
# 6) Evaluate a model with any chosen saved mask file
#
# Consistency guarantee:
#   train_idx / test_idx are created ONCE and reused everywhere
# ============================================================

import os
import json
import random
import numpy as np
import pandas as pd
import torch

from collections import defaultdict
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    mean_squared_error,
    mean_absolute_error,
)
from transformers import AutoTokenizer
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction


# ============================================================
# 0. REPRODUCIBILITY
# ============================================================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed(42)


# ============================================================
# 1. CONFIG
# ============================================================

DATA_CSV = "all_data_after2_updated.csv"
TEXT_EMB_PATH = "textual_embeddings.npy"
CAT_EMB_PATH = "categorical_embeddings_reduced.npy"
NUM_EMB_PATH = "numerical_embeddings_reduced.npy"

OUT_DIR = "fixed_eval_setup_test_ratio_multi_13cat_9num_1"
FIXED_SETUP_PATH = os.path.join(OUT_DIR, "fixed_setup.json")
MASKS_DIR = os.path.join(OUT_DIR, "masks")

TEST_RATIO = 0.3
RANDOM_STATE = 42
NUM_MASKS_PER_SCENARIO = 5

# Same fixed test set, but generate masks for these missing ratios
MISSING_RATIOS = [0.1, 0.2, 0.3, 0.4, 0.5]

SCENARIOS = ("MCAR", "MAR", "MNAR")


# ============================================================
# 2. LOAD TOKENIZER
# ============================================================

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")


# ============================================================
# 3. LOAD DATA
# ============================================================

df = pd.read_csv(DATA_CSV, dtype=str)

textual_embeddings = np.load(TEXT_EMB_PATH, allow_pickle=True)
categorical_embeddings = np.load(CAT_EMB_PATH, allow_pickle=True)
numerical_embeddings = np.load(NUM_EMB_PATH, allow_pickle=True)

print("Textual Embeddings Shape:", textual_embeddings.shape)
print("Categorical Embeddings Shape:", categorical_embeddings.shape)
print("Numerical Embeddings Shape:", numerical_embeddings.shape)

all_embeddings_np = np.concatenate(
    [textual_embeddings, categorical_embeddings, numerical_embeddings], axis=1
)
print("Combined Embeddings Shape:", all_embeddings_np.shape)

normalized_embeddings = torch.tensor(all_embeddings_np, dtype=torch.float32)


# ============================================================
# 4. DEFINE COLUMNS / MODALITIES
# ============================================================

textual_columns = [
    "Event full description",
    "Lesson Learnt",
    "Post-event summary",
]
categorical_indices_raw = [1, 2, 3, 8, 21, 22, 23,  27, 29, 31, 35, 36, 37]

#categorical_indices_raw = [1, 2, 3,  8,  15,  21, 22, 23,  27, 29, 31, 35, 36, 37]
numerical_indices_raw = [10,  13, 14, 20,  32, 33, 34, 39, 40]
#numerical_indices_raw = [10, 11, 12, 13, 14, 16, 17, 18, 20, 26, 32, 33, 34, 39, 40]

categorical_columns = [df.columns[i] for i in categorical_indices_raw]
numerical_columns = [df.columns[i] for i in numerical_indices_raw]

# Exactly 38 total attributes
modalities = ['txt'] * 3 + ['cat'] * 13 + ['num'] * 9

print("Num textual attrs:", len(textual_columns))
print("Num categorical attrs:", len(categorical_columns))
print("Num numerical attrs:", len(numerical_columns))
print("Total attrs:", len(modalities))


# ============================================================
# 5. BUILD GROUND TRUTH + RAW VALUES
# ============================================================

def build_ground_truth(df, textual_columns, categorical_columns, numerical_columns):
    # ----- Text -----
    ground_truth_text = []
    for col in textual_columns:
        values = df[col].fillna("").astype(str).values
        values = np.array([
            "" if (str(v).strip() == "" or str(v).strip().lower() == "nan") else str(v)
            for v in values
        ], dtype=object)
        ground_truth_text.append(values)

    ground_truth_text = np.array(ground_truth_text, dtype=object).T  # [N, 3]

    # ----- Categorical -----
    cat_maps = {}
    ground_truth_cat = []
    raw_categorical_for_mar = []

    for col in categorical_columns:
        values = df[col].astype(str)
        values = values.fillna("")

        missing_mask = (
            values.isna() |
            (values.str.strip() == "") |
            (values.str.strip().str.lower() == "nan")
        )

        encoded = np.full(len(values), -1, dtype=int)

        non_missing_values = values[~missing_mask]
        le = LabelEncoder()

        if len(non_missing_values) > 0:
            le.fit(non_missing_values)
            encoded[~missing_mask] = le.transform(non_missing_values)

        ground_truth_cat.append(encoded)
        raw_categorical_for_mar.append(encoded.copy())
        cat_maps[col] = le

    ground_truth_cat = np.stack(ground_truth_cat, axis=1)  # [N, 20]
    raw_categorical_for_mar = np.stack(raw_categorical_for_mar, axis=1)

    # ----- Numerical -----
    num_df = df[numerical_columns].apply(pd.to_numeric, errors="coerce")
    raw_numerical_for_mar = num_df.values.astype(float)

    scaler = StandardScaler()
    filled_for_scaling = num_df.fillna(num_df.mean())
    ground_truth_num = scaler.fit_transform(filled_for_scaling)
    ground_truth_num[num_df.isna().values] = -1

    # ----- Combine -----
    final_ground_truth = np.concatenate(
        [
            ground_truth_text,
            ground_truth_cat.astype(object),
            ground_truth_num.astype(object),
        ],
        axis=1
    )

    return final_ground_truth, cat_maps, scaler, raw_numerical_for_mar, raw_categorical_for_mar


(
    ground_truth,
    cat_maps,
    num_scaler,
    numerical_values,
    categorical_indices,
) = build_ground_truth(
    df=df,
    textual_columns=textual_columns,
    categorical_columns=categorical_columns,
    numerical_columns=numerical_columns,
)

print("Final Ground Truth Shape:", ground_truth.shape)
print("Raw numerical values shape:", numerical_values.shape)
print("Raw categorical indices shape:", categorical_indices.shape)


# ============================================================
# 6. HELPERS
# ============================================================

def build_gt_missing_mask(gt_row):
    missing_mask = []
    for x in gt_row:
        if isinstance(x, str):
            x_str = x.strip()
            missing_mask.append(
                x_str == "" or x_str == "-1" or x_str.lower() == "nan"
            )
        else:
            try:
                missing_mask.append(float(x) == -1)
            except Exception:
                missing_mask.append(False)
    return missing_mask


def get_observed_embedding_indices(emb, eps=1e-6):
    if torch.is_tensor(emb):
        non_zero_mask = emb.abs().sum(dim=1) > eps
        return non_zero_mask.nonzero(as_tuple=True)[0].tolist()
    else:
        emb = np.asarray(emb)
        return np.where(np.abs(emb).sum(axis=1) > eps)[0].tolist()


def safe_text_ids(x, tokenizer):
    if isinstance(x, str):
        x = x.strip()
        if x == "" or x == "-1" or x.lower() == "nan":
            return []
        return tokenizer.encode(x, add_special_tokens=False)
    elif isinstance(x, torch.Tensor):
        return x.view(-1).tolist()
    elif isinstance(x, (list, tuple, np.ndarray)):
        return list(x)
    else:
        try:
            return [int(x)]
        except Exception:
            return []


def decode_ids_clean(ids, tokenizer):
    ignore_tokens = {0, 101, 102}
    ids = [int(t) for t in ids if int(t) not in ignore_tokens]
    return tokenizer.decode(ids, skip_special_tokens=True), ids


def safe_float(x, default=0.0):
    try:
        return float(x)
    except Exception:
        return default


def text_length_score(x):
    if isinstance(x, str):
        s = x.strip()
        if s == "" or s == "-1" or s.lower() == "nan":
            return 0.0
        return float(len(s.split()))
    return 0.0


def numeric_score_from_raw(sample_idx, numerical_values, categorical_indices):
    vals = []

    if numerical_values is not None:
        row_num = numerical_values[sample_idx]
        if torch.is_tensor(row_num):
            row_num = row_num.detach().cpu().numpy()
        for v in row_num:
            try:
                fv = float(v)
                if fv != -1 and not np.isnan(fv):
                    vals.append(abs(fv))
            except Exception:
                pass

    if categorical_indices is not None:
        row_cat = categorical_indices[sample_idx]
        if torch.is_tensor(row_cat):
            row_cat = row_cat.detach().cpu().numpy()
        for v in row_cat:
            try:
                fv = float(v)
                if fv != -1 and not np.isnan(fv):
                    vals.append(abs(fv))
            except Exception:
                pass

    if len(vals) == 0:
        return 0.5

    return float(np.mean(vals))


def get_attr_value_score(
    attr_idx,
    gt_i,
    sample_idx,
    modalities,
    missing_scenario,
    numerical_values=None,
    categorical_indices=None,
):
    mod = modalities[attr_idx]
    scenario = missing_scenario.upper()

    if scenario == "MNAR":
        if mod == "num":
            return abs(safe_float(gt_i, 0.0))
        elif mod == "cat":
            return float(safe_float(gt_i, 0.0))
        elif mod == "txt":
            return text_length_score(gt_i)
        return 0.0

    elif scenario == "MAR":
        sample_score = numeric_score_from_raw(
            sample_idx=sample_idx,
            numerical_values=numerical_values,
            categorical_indices=categorical_indices
        )
        attr_bias = (attr_idx + 1) / (len(modalities) + 1)
        return sample_score + 0.1 * attr_bias

    elif scenario == "MCAR":
        return 1.0

    else:
        raise ValueError("missing_scenario must be 'MCAR', 'MAR', or 'MNAR'")


def normalize_scores(scores):
    scores = np.array(scores, dtype=np.float64)

    if len(scores) == 0:
        return scores

    if np.allclose(scores, scores[0]):
        return np.ones_like(scores) / len(scores)

    scores = (scores - scores.min()) / (scores.max() - scores.min() + 1e-12)
    scores = scores + 1e-6
    scores = scores / scores.sum()
    return scores


def weighted_sample_without_replacement(items, weights, k, np_rng):
    if len(items) == 0 or k <= 0:
        return []

    k = min(k, len(items))
    weights = np.array(weights, dtype=np.float64)
    weights = weights / weights.sum()
    chosen_idx = np_rng.choice(len(items), size=k, replace=False, p=weights)
    return [items[i] for i in chosen_idx]


def build_missing_ratio_dict(ratio):
    ratio = float(ratio)
    ratio = max(0.0, min(1.0, ratio))
    return {
        "txt": ratio,
        "cat": ratio,
        "num": ratio
    }


def ratio_to_folder_name(ratio):
    return f"ratio_{float(ratio):.1f}"


# ============================================================
# 7. CREATE OR LOAD ONE FIXED SPLIT
# ============================================================

def create_or_load_fixed_setup(
    normalized_embeddings,
    ground_truth,
    test_ratio=0.3,
    random_state=42,
    out_path=FIXED_SETUP_PATH,
    force_rebuild=False
):
    os.makedirs(os.path.dirname(out_path), exist_ok=True)

    if os.path.exists(out_path) and not force_rebuild:
        with open(out_path, "r") as f:
            setup = json.load(f)
        print("\nLoaded existing fixed setup from:", out_path)
        print("Train size:", len(setup["train_idx"]))
        print("Test size:", len(setup["test_idx"]))
        return setup

    indices = np.arange(len(normalized_embeddings))
    train_idx, test_idx = train_test_split(
        indices,
        test_size=test_ratio,
        random_state=random_state
    )

    setup = {
        "random_state": random_state,
        "test_ratio": test_ratio,
        "train_idx": train_idx.tolist(),
        "test_idx": test_idx.tolist(),
        "test_samples": {}
    }

    print("\n===== BUILDING FIXED SETUP =====")
    print("Train size:", len(train_idx))
    print("Test size:", len(test_idx))

    for sample_idx in test_idx:
        emb = normalized_embeddings[sample_idx]
        gt = ground_truth[sample_idx]

        gt_missing_mask = build_gt_missing_mask(gt)
        non_missing_gt = [i for i, m in enumerate(gt_missing_mask) if not m]

        observed_emb = get_observed_embedding_indices(emb)

        usable = sorted(set(non_missing_gt).intersection(observed_emb))

        setup["test_samples"][str(int(sample_idx))] = {
            "non_missing_gt": non_missing_gt,
            "observed_embedding": observed_emb,
            "usable_indices": usable
        }

    with open(out_path, "w") as f:
        json.dump(setup, f, indent=2)

    print("Saved fixed setup to:", out_path)
    return setup


def print_fixed_setup_preview(setup, max_samples=10):
    print("\n===== FIXED SETUP PREVIEW =====")
    print("Train size:", len(setup["train_idx"]))
    print("Test size:", len(setup["test_idx"]))

    shown = 0
    for sample_idx, info in setup["test_samples"].items():
        print(f"\nSample {sample_idx}")
        print("  non_missing_gt     =", info["non_missing_gt"])
        print("  observed_embedding =", info["observed_embedding"])
        print("  usable_indices     =", info["usable_indices"])
        shown += 1
        if shown >= max_samples:
            break


# ============================================================
# 8. GENERATE MASKS FOR MCAR / MAR / MNAR USING SAME SPLIT
#    AT MULTIPLE MISSING RATIOS
# ============================================================

def choose_masked_indices_from_candidates_with_scenario(
    sample_idx,
    candidate_indices,
    ground_truth,
    modalities,
    missing_scenario="MCAR",
    missing_ratio_per_modality=None,
    numerical_values=None,
    categorical_indices=None,
    rng=None,
    np_rng=None
):
    if rng is None:
        rng = random.Random(42)
    if np_rng is None:
        np_rng = np.random.default_rng(42)

    if missing_ratio_per_modality is None:
        missing_ratio_per_modality = {
            "txt": 0.3,
            "cat": 0.3,
            "num": 0.3
        }

    masked = []

    for mod in ["txt", "cat", "num"]:
        pool = [idx for idx in candidate_indices if modalities[idx] == mod]

        if len(pool) == 0:
            continue

        ratio = float(missing_ratio_per_modality.get(mod, 0.0))
        ratio = max(0.0, min(1.0, ratio))

        if ratio == 0.0:
            continue

        k = int(np.ceil(len(pool) * ratio))
        k = max(1, k)
        k = min(k, len(pool))

        scenario = missing_scenario.upper()

        if scenario == "MCAR":
            chosen = rng.sample(pool, k)

        elif scenario in {"MAR", "MNAR"}:
            gt_row = ground_truth[sample_idx]
            scores = [
                get_attr_value_score(
                    attr_idx=attr_idx,
                    gt_i=gt_row[attr_idx],
                    sample_idx=sample_idx,
                    modalities=modalities,
                    missing_scenario=scenario,
                    numerical_values=numerical_values,
                    categorical_indices=categorical_indices
                )
                for attr_idx in pool
            ]
            probs = normalize_scores(scores)
            chosen = weighted_sample_without_replacement(pool, probs, k, np_rng)

        else:
            raise ValueError("missing_scenario must be 'MCAR', 'MAR', or 'MNAR'")

        masked.extend(chosen)

    return sorted(set(masked))


def generate_all_fixed_masks(
    setup,
    modalities,
    ground_truth,
    numerical_values,
    categorical_indices,
    out_dir=MASKS_DIR,
    scenarios=("MCAR", "MAR", "MNAR"),
    missing_ratios=(0.1, 0.2, 0.3, 0.4, 0.5),
    num_masks_per_scenario=10,
    base_seed=42,
    force_regenerate=False
):
    os.makedirs(out_dir, exist_ok=True)

    all_paths = []

    scenario_seed_offset = {
        "MCAR": 0,
        "MAR": 100000,
        "MNAR": 200000
    }

    for scenario in scenarios:
        scenario = scenario.upper()
        scenario_dir = os.path.join(out_dir, scenario.lower())
        os.makedirs(scenario_dir, exist_ok=True)

        for ratio in missing_ratios:
            ratio = float(ratio)
            ratio_dir_name = ratio_to_folder_name(ratio)
            ratio_dir = os.path.join(scenario_dir, ratio_dir_name)
            os.makedirs(ratio_dir, exist_ok=True)

            missing_ratio_per_modality = build_missing_ratio_dict(ratio)

            for mask_id in range(num_masks_per_scenario):
                path = os.path.join(ratio_dir, f"mask_{mask_id}.json")

                if os.path.exists(path) and not force_regenerate:
                    print("Exists, keeping:", path)
                    all_paths.append(path)
                    continue

                # Stable reproducible seed per scenario + ratio + mask_id
                ratio_offset = int(round(ratio * 1000))
                seed = base_seed + scenario_seed_offset[scenario] + ratio_offset + mask_id

                rng = random.Random(seed)
                np_rng = np.random.default_rng(seed)

                mask_dict = {}

                for sample_idx_str, info in setup["test_samples"].items():
                    sample_idx = int(sample_idx_str)
                    candidate_indices = info["usable_indices"]

                    masked = choose_masked_indices_from_candidates_with_scenario(
                        sample_idx=sample_idx,
                        candidate_indices=candidate_indices,
                        ground_truth=ground_truth,
                        modalities=modalities,
                        missing_scenario=scenario,
                        missing_ratio_per_modality=missing_ratio_per_modality,
                        numerical_values=numerical_values,
                        categorical_indices=categorical_indices,
                        rng=rng,
                        np_rng=np_rng
                    )

                    mask_dict[sample_idx_str] = masked

                payload = {
                    "mask_id": mask_id,
                    "missing_scenario": scenario,
                    "missing_ratio": ratio,
                    "ratio_folder": ratio_dir_name,
                    "train_idx": setup["train_idx"],
                    "test_idx": setup["test_idx"],
                    "missing_ratio_per_modality": missing_ratio_per_modality,
                    "num_total_attributes": len(modalities),
                    "modalities_summary": {
                        "txt": modalities.count("txt"),
                        "cat": modalities.count("cat"),
                        "num": modalities.count("num")
                    },
                    "masks": mask_dict
                }

                with open(path, "w") as f:
                    json.dump(payload, f, indent=2)

                print("Saved:", path)
                all_paths.append(path)

    return all_paths


def print_mask_summary(mask_file_path, modalities, max_samples=10):
    with open(mask_file_path, "r") as f:
        mask_data = json.load(f)

    print("\n===== MASK SUMMARY =====")
    print("Mask file:", mask_file_path)
    print("Scenario:", mask_data.get("missing_scenario"))
    print("Missing ratio:", mask_data.get("missing_ratio"))
    print("Missing ratio per modality:", mask_data.get("missing_ratio_per_modality"))

    shown = 0
    for sample_idx, masked_indices in mask_data["masks"].items():
        txt_masked = [i for i in masked_indices if modalities[i] == "txt"]
        cat_masked = [i for i in masked_indices if modalities[i] == "cat"]
        num_masked = [i for i in masked_indices if modalities[i] == "num"]

        print(f"Sample {sample_idx}")
        print("  txt masked:", txt_masked)
        print("  cat masked:", cat_masked)
        print("  num masked:", num_masked)

        shown += 1
        if shown >= max_samples:
            break


# ============================================================
# 9. EVALUATE WITH ONE FIXED MASK FILE
# ============================================================

def evaluate_per_sample_with_fixed_mask(
    model,
    normalized_embeddings,
    ground_truth,
    modalities,
    edge_index,
    numerical_values,
    categorical_indices,
    tokenizer,
    mask_file_path
):
    model.eval()

    with open(mask_file_path, "r") as f:
        mask_payload = json.load(f)

    fixed_masks = mask_payload["masks"]
    test_idx = [int(x) for x in mask_payload["test_idx"]]

    total_eval_loss = 0.0
    total_valid_count = 0

    num_targets_by_attr = defaultdict(list)
    num_preds_by_attr = defaultdict(list)

    cat_targets_by_attr = defaultdict(list)
    cat_preds_by_attr = defaultdict(list)

    txt_token_correct_by_attr = defaultdict(int)
    txt_token_total_by_attr = defaultdict(int)

    bleu_references_by_attr = defaultdict(list)
    bleu_hypotheses_by_attr = defaultdict(list)

    smooth = SmoothingFunction().method4

    with torch.no_grad():
        for sample_idx in test_idx:
            sample_idx_str = str(int(sample_idx))
            if sample_idx_str not in fixed_masks:
                continue

            masked_attr_indices = fixed_masks[sample_idx_str]
            if len(masked_attr_indices) == 0:
                continue

            emb = normalized_embeddings[sample_idx].clone()
            gt = ground_truth[sample_idx]

            missing_mask = build_gt_missing_mask(gt)
            missing_mask = torch.tensor(
                missing_mask, dtype=torch.bool, device=emb.device
            )

            emb_masked = emb.clone()
            mask_copy = missing_mask.clone()

            for masked_attr_idx in masked_attr_indices:
                emb_masked[masked_attr_idx] = 0.0
                mask_copy[masked_attr_idx] = True

            loss, outputs = model(
                llm_embeddings=emb_masked,
                raw_numericals=numerical_values[sample_idx],
                cat_indices=categorical_indices[sample_idx],
                missing_mask=mask_copy,
                edge_index=edge_index,
                modality_types=modalities,
                targets=gt,
                attr_indices=list(range(len(modalities))),
                mode='test'
            )

            total_eval_loss += loss.item()

            for masked_attr_idx in masked_attr_indices:
                mod = modalities[masked_attr_idx]
                gt_i = gt[masked_attr_idx]
                out_i = outputs[masked_attr_idx]

                if mod == 'num':
                    try:
                        gt_val = float(gt_i)
                    except Exception:
                        continue

                    if gt_val == -1:
                        continue

                    if torch.is_tensor(out_i):
                        pred_val = out_i.squeeze().item()
                    else:
                        pred_val = float(out_i)

                    num_targets_by_attr[masked_attr_idx].append(gt_val)
                    num_preds_by_attr[masked_attr_idx].append(pred_val)
                    total_valid_count += 1

                elif mod == 'cat':
                    try:
                        gt_val = int(gt_i)
                    except Exception:
                        continue

                    if gt_val == -1:
                        continue

                    if torch.is_tensor(out_i):
                        if out_i.dim() == 2:
                            pred_val = out_i.argmax(dim=1).item()
                        else:
                            pred_val = out_i.argmax().item()
                    else:
                        pred_val = int(out_i)

                    cat_targets_by_attr[masked_attr_idx].append(gt_val)
                    cat_preds_by_attr[masked_attr_idx].append(pred_val)
                    total_valid_count += 1

                elif mod == 'txt':
                    if isinstance(gt_i, str):
                        if len(gt_i.strip()) == 0 or gt_i.strip() == "-1":
                            continue
                        gt_ids = tokenizer.encode(gt_i, add_special_tokens=False)
                    elif isinstance(gt_i, torch.Tensor):
                        gt_ids = gt_i.view(-1).tolist()
                    elif isinstance(gt_i, (list, tuple)):
                        gt_ids = list(gt_i)
                    else:
                        gt_ids = [int(gt_i)]

                    if out_i.dim() == 3:
                        pred_ids = out_i.argmax(dim=-1).squeeze(0).tolist()
                    elif out_i.dim() == 2:
                        pred_ids = out_i.argmax(dim=-1).tolist()
                    else:
                        raise ValueError(f"Unexpected text output shape: {out_i.shape}")

                    ignore_tokens = {0, 101, 102}
                    gt_ids = [int(t) for t in gt_ids if int(t) not in ignore_tokens]
                    pred_ids = [int(t) for t in pred_ids if int(t) not in ignore_tokens]

                    min_len = min(len(gt_ids), len(pred_ids))
                    if min_len > 0:
                        txt_token_correct_by_attr[masked_attr_idx] += sum(
                            g == p for g, p in zip(gt_ids[:min_len], pred_ids[:min_len])
                        )
                        txt_token_total_by_attr[masked_attr_idx] += min_len

                    ref_sentence = tokenizer.decode(gt_ids, skip_special_tokens=True)
                    hyp_sentence = tokenizer.decode(pred_ids, skip_special_tokens=True)

                    ref_words = ref_sentence.split()
                    hyp_words = hyp_sentence.split()

                    if len(ref_words) > 0 and len(hyp_words) > 0:
                        bleu_references_by_attr[masked_attr_idx].append([ref_words])
                        bleu_hypotheses_by_attr[masked_attr_idx].append(hyp_words)

                    total_valid_count += 1

    results = {}

    num_mae_per_attr = {}
    num_mse_per_attr = {}
    num_valid_per_attr = {}

    for attr_idx in sorted(num_targets_by_attr.keys()):
        y_true = num_targets_by_attr[attr_idx]
        y_pred = num_preds_by_attr[attr_idx]

        if len(y_true) == 0:
            continue

        num_mae_per_attr[attr_idx] = mean_absolute_error(y_true, y_pred)
        num_mse_per_attr[attr_idx] = mean_squared_error(y_true, y_pred)
        num_valid_per_attr[attr_idx] = len(y_true)

    if len(num_valid_per_attr) > 0:
        total_num_valid = sum(num_valid_per_attr.values())
        results["num_mae_weighted"] = sum(
            num_mae_per_attr[a] * num_valid_per_attr[a] for a in num_mae_per_attr
        ) / total_num_valid
        results["num_mse_weighted"] = sum(
            num_mse_per_attr[a] * num_valid_per_attr[a] for a in num_mse_per_attr
        ) / total_num_valid
        results["num_mae_macro_attr"] = np.mean(list(num_mae_per_attr.values()))
        results["num_mse_macro_attr"] = np.mean(list(num_mse_per_attr.values()))
        results["num_mae_per_attr"] = num_mae_per_attr
        results["num_mse_per_attr"] = num_mse_per_attr
        results["num_valid_per_attr"] = num_valid_per_attr

    cat_f1_weighted_per_attr = {}
    cat_f1_macro_per_attr = {}
    cat_acc_per_attr = {}
    cat_valid_per_attr = {}

    for attr_idx in sorted(cat_targets_by_attr.keys()):
        y_true = cat_targets_by_attr[attr_idx]
        y_pred = cat_preds_by_attr[attr_idx]

        if len(y_true) == 0:
            continue

        cat_f1_weighted_per_attr[attr_idx] = f1_score(
            y_true, y_pred, average="weighted", zero_division=0
        )
        cat_f1_macro_per_attr[attr_idx] = f1_score(
            y_true, y_pred, average="macro", zero_division=0
        )
        cat_acc_per_attr[attr_idx] = accuracy_score(y_true, y_pred)
        cat_valid_per_attr[attr_idx] = len(y_true)

    if len(cat_valid_per_attr) > 0:
        total_cat_valid = sum(cat_valid_per_attr.values())
        results["cat_f1_weighted_attr_weighted"] = sum(
            cat_f1_weighted_per_attr[a] * cat_valid_per_attr[a]
            for a in cat_f1_weighted_per_attr
        ) / total_cat_valid
        results["cat_f1_macro_attr_weighted"] = sum(
            cat_f1_macro_per_attr[a] * cat_valid_per_attr[a]
            for a in cat_f1_macro_per_attr
        ) / total_cat_valid
        results["cat_acc_weighted"] = sum(
            cat_acc_per_attr[a] * cat_valid_per_attr[a]
            for a in cat_acc_per_attr
        ) / total_cat_valid
        results["cat_f1_weighted_macro_attr"] = np.mean(list(cat_f1_weighted_per_attr.values()))
        results["cat_f1_macro_macro_attr"] = np.mean(list(cat_f1_macro_per_attr.values()))
        results["cat_acc_macro_attr"] = np.mean(list(cat_acc_per_attr.values()))
        results["cat_f1_weighted_per_attr"] = cat_f1_weighted_per_attr
        results["cat_f1_macro_per_attr"] = cat_f1_macro_per_attr
        results["cat_acc_per_attr"] = cat_acc_per_attr
        results["cat_valid_per_attr"] = cat_valid_per_attr

    txt_token_acc_per_attr = {}
    txt_bleu_per_attr = {}
    txt_valid_per_attr = {}

    txt_attr_ids = sorted(set(list(txt_token_total_by_attr.keys()) + list(bleu_references_by_attr.keys())))

    for attr_idx in txt_attr_ids:
        token_total = txt_token_total_by_attr.get(attr_idx, 0)
        refs = bleu_references_by_attr.get(attr_idx, [])
        hyps = bleu_hypotheses_by_attr.get(attr_idx, [])

        txt_token_acc_per_attr[attr_idx] = (
            txt_token_correct_by_attr[attr_idx] / token_total if token_total > 0 else 0.0
        )

        txt_bleu_per_attr[attr_idx] = (
            corpus_bleu(refs, hyps, smoothing_function=smooth) if len(refs) > 0 else 0.0
        )

        txt_valid_per_attr[attr_idx] = max(len(refs), 0)

    if len(txt_valid_per_attr) > 0 and sum(txt_valid_per_attr.values()) > 0:
        total_txt_valid = sum(txt_valid_per_attr.values())

        results["txt_token_acc_weighted"] = sum(
            txt_token_acc_per_attr[a] * txt_valid_per_attr[a]
            for a in txt_token_acc_per_attr
        ) / total_txt_valid

        results["txt_bleu_weighted"] = sum(
            txt_bleu_per_attr[a] * txt_valid_per_attr[a]
            for a in txt_bleu_per_attr
        ) / total_txt_valid

        results["txt_token_acc_macro_attr"] = np.mean(list(txt_token_acc_per_attr.values()))
        results["txt_bleu_macro_attr"] = np.mean(list(txt_bleu_per_attr.values()))

        results["txt_token_acc_per_attr"] = txt_token_acc_per_attr
        results["txt_bleu_per_attr"] = txt_bleu_per_attr
        results["txt_valid_per_attr"] = txt_valid_per_attr
    else:
        results["txt_token_acc_weighted"] = 0.0
        results["txt_bleu_weighted"] = 0.0

    results["total_loss"] = total_eval_loss / max(total_valid_count, 1)
    results["mask_file_path"] = mask_file_path
    results["missing_scenario"] = mask_payload.get("missing_scenario")
    results["missing_ratio"] = mask_payload.get("missing_ratio")
    results["train_size"] = len(mask_payload["train_idx"])
    results["test_size"] = len(mask_payload["test_idx"])

    print("\n===== EVALUATION RESULTS =====")
    print(results)
    return results


# ============================================================
# 10. OPTIONAL: EVALUATE ALL SAVED MASKS
# ============================================================

def evaluate_all_saved_masks(
    model,
    normalized_embeddings,
    ground_truth,
    modalities,
    edge_index,
    numerical_values,
    categorical_indices,
    tokenizer,
    masks_dir=MASKS_DIR,
    scenarios=("MCAR", "MAR", "MNAR"),
    missing_ratios=(0.1, 0.2, 0.3, 0.4, 0.5),
    num_masks_per_scenario=10
):
    all_results = {}

    for scenario in scenarios:
        scenario_key = scenario.lower()
        all_results[scenario_key] = {}

        for ratio in missing_ratios:
            ratio_str = f"{float(ratio):.1f}"
            all_results[scenario_key][ratio_str] = []

            for mask_id in range(num_masks_per_scenario):
                mask_path = os.path.join(
                    masks_dir,
                    scenario_key,
                    ratio_to_folder_name(ratio),
                    f"mask_{mask_id}.json"
                )

                if not os.path.exists(mask_path):
                    print("Skipping missing mask:", mask_path)
                    continue

                metrics = evaluate_per_sample_with_fixed_mask(
                    model=model,
                    normalized_embeddings=normalized_embeddings,
                    ground_truth=ground_truth,
                    modalities=modalities,
                    edge_index=edge_index,
                    numerical_values=numerical_values,
                    categorical_indices=categorical_indices,
                    tokenizer=tokenizer,
                    mask_file_path=mask_path
                )
                all_results[scenario_key][ratio_str].append(metrics)

    return all_results


def summarize_all_results(all_results):
    summary_rows = []

    for scenario, ratio_dict in all_results.items():
        for ratio_str, metrics_list in ratio_dict.items():
            if len(metrics_list) == 0:
                continue

            row = {
                "scenario": scenario,
                "missing_ratio": ratio_str,
                "num_runs": len(metrics_list),
            }

            numeric_keys = [
                "total_loss",
                "num_mae_weighted",
                "num_mse_weighted",
                "num_mae_macro_attr",
                "num_mse_macro_attr",
                "cat_f1_weighted_attr_weighted",
                "cat_f1_macro_attr_weighted",
                "cat_acc_weighted",
                "cat_f1_weighted_macro_attr",
                "cat_f1_macro_macro_attr",
                "cat_acc_macro_attr",
                "txt_token_acc_weighted",
                "txt_bleu_weighted",
                "txt_token_acc_macro_attr",
                "txt_bleu_macro_attr",
            ]

            for key in numeric_keys:
                vals = [m[key] for m in metrics_list if key in m and isinstance(m[key], (int, float, np.floating))]
                if len(vals) > 0:
                    row[f"{key}_mean"] = float(np.mean(vals))
                    row[f"{key}_std"] = float(np.std(vals))

            summary_rows.append(row)

    summary_df = pd.DataFrame(summary_rows)

    if len(summary_df) > 0:
        summary_df = summary_df.sort_values(by=["scenario", "missing_ratio"]).reset_index(drop=True)

    return summary_df


# ============================================================
# 11. ONE MASTER FUNCTION
# ============================================================

def prepare_fixed_evaluation_artifacts(
    normalized_embeddings,
    ground_truth,
    modalities,
    numerical_values,
    categorical_indices,
    test_ratio=0.2,
    random_state=42,
    fixed_setup_path=FIXED_SETUP_PATH,
    masks_dir=MASKS_DIR,
    missing_ratios=(0.1, 0.2, 0.3, 0.4, 0.5),
    scenarios=("MCAR", "MAR", "MNAR"),
    num_masks_per_scenario=10,
    force_rebuild_setup=False,
    force_regenerate_masks=False
):
    setup = create_or_load_fixed_setup(
        normalized_embeddings=normalized_embeddings,
        ground_truth=ground_truth,
        test_ratio=test_ratio,
        random_state=random_state,
        out_path=fixed_setup_path,
        force_rebuild=force_rebuild_setup
    )

    print_fixed_setup_preview(setup, max_samples=10)

    mask_paths = generate_all_fixed_masks(
        setup=setup,
        modalities=modalities,
        ground_truth=ground_truth,
        numerical_values=numerical_values,
        categorical_indices=categorical_indices,
        out_dir=masks_dir,
        scenarios=scenarios,
        missing_ratios=missing_ratios,
        num_masks_per_scenario=num_masks_per_scenario,
        base_seed=random_state,
        force_regenerate=force_regenerate_masks
    )

    return setup, mask_paths


# ============================================================
# 12. RUN ARTIFACT PREPARATION ONCE
# ============================================================

setup, mask_paths = prepare_fixed_evaluation_artifacts(
    normalized_embeddings=normalized_embeddings,
    ground_truth=ground_truth,
    modalities=modalities,
    numerical_values=numerical_values,
    categorical_indices=categorical_indices,
    test_ratio=TEST_RATIO,
    random_state=RANDOM_STATE,
    fixed_setup_path=FIXED_SETUP_PATH,
    masks_dir=MASKS_DIR,
    missing_ratios=MISSING_RATIOS,
    scenarios=SCENARIOS,
    num_masks_per_scenario=NUM_MASKS_PER_SCENARIO,
    force_rebuild_setup=False,
    force_regenerate_masks=False
)

# Example preview of one mask
example_mask = os.path.join(MASKS_DIR, "mcar", "ratio_0.3", "mask_0.json")
if os.path.exists(example_mask):
    print_mask_summary(example_mask, modalities, max_samples=10)

print("\nArtifacts ready.")
print("Fixed setup:", FIXED_SETUP_PATH)
print("Masks dir:", MASKS_DIR)
print(f"Total mask files prepared/found: {len(mask_paths)}")


# ============================================================
# 13. EXAMPLE EVALUATION USAGE
# ============================================================

# Example: evaluate one mask file
# metrics = evaluate_per_sample_with_fixed_mask(
#     model=model,
#     normalized_embeddings=normalized_embeddings,
#     ground_truth=ground_truth,
#     modalities=modalities,
#     edge_index=edge_index,
#     numerical_values=numerical_values,
#     categorical_indices=categorical_indices,
#     tokenizer=tokenizer,
#     mask_file_path=os.path.join(MASKS_DIR, "mcar", "ratio_0.3", "mask_0.json")
# )

# Example: evaluate all scenario-ratio-mask combinations
# all_results = evaluate_all_saved_masks(
#     model=model,
#     normalized_embeddings=normalized_embeddings,
#     ground_truth=ground_truth,
#     modalities=modalities,
#     edge_index=edge_index,
#     numerical_values=numerical_values,
#     categorical_indices=categorical_indices,
#     tokenizer=tokenizer,
#     masks_dir=MASKS_DIR,
#     scenarios=SCENARIOS,
#     missing_ratios=MISSING_RATIOS,
#     num_masks_per_scenario=NUM_MASKS_PER_SCENARIO
# )

# summary_df = summarize_all_results(all_results)
# print(summary_df)

# Optional save summary
# summary_df.to_csv(os.path.join(OUT_DIR, "evaluation_summary.csv"), index=False)

Textual Embeddings Shape: (755, 3, 384)
Categorical Embeddings Shape: (755, 13, 384)
Numerical Embeddings Shape: (755, 9, 384)
Combined Embeddings Shape: (755, 25, 384)
Num textual attrs: 3
Num categorical attrs: 13
Num numerical attrs: 9
Total attrs: 25


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Final Ground Truth Shape: (755, 25)
Raw numerical values shape: (755, 9)
Raw categorical indices shape: (755, 13)

===== BUILDING FIXED SETUP =====
Train size: 528
Test size: 227
Saved fixed setup to: fixed_eval_setup_test_ratio_multi_13cat_9num_1/fixed_setup.json

===== FIXED SETUP PREVIEW =====
Train size: 528
Test size: 227

Sample 291
  non_missing_gt     = [0, 1, 3, 4, 5, 6, 10, 11, 12, 13, 14, 15, 16, 23, 24]
  observed_embedding = [0, 1, 3, 4, 5, 6, 10, 11, 12, 13, 14, 15, 16, 23, 24]
  usable_indices     = [0, 1, 3, 4, 5, 6, 10, 11, 12, 13, 14, 15, 16, 23, 24]

Sample 536
  non_missing_gt     = [0, 2, 3, 4, 5, 10, 11, 12, 13, 14, 23, 24]
  observed_embedding = [0, 2, 3, 4, 5, 10, 11, 12, 13, 14, 23, 24]
  usable_indices     = [0, 2, 3, 4, 5, 10, 11, 12, 13, 14, 23, 24]

Sample 39
  non_missing_gt     = [0, 2, 3, 4, 5, 10, 11, 12, 14, 15]
  observed_embedding = [0, 2, 3, 4, 5, 10, 11, 12, 14, 15]
  usable_indices     = [0, 2, 3, 4, 5, 10, 11, 12, 14, 15]

Sample 77
  non_missing

In [ ]:

# ============================================================
# 15. EVALUATION
# ============================================================

def ratio_to_folder_name(ratio):
    return f"ratio_{float(ratio):.1f}"

def evaluate_mask_file(model, mask_file_path):
    with open(mask_file_path, "r") as f:
        payload = json.load(f)

    masks = payload["masks"]
    eval_rows = [int(x) for x in payload["test_idx"]]

    cat_y_true, cat_y_pred = [], []
    num_abs_errors = []

    per_attr_num_errors = defaultdict(list)
    per_attr_cat_true = defaultdict(list)
    per_attr_cat_pred = defaultdict(list)

    for r in eval_rows:
        key = str(int(r))
        if key not in masks:
            continue

        masked_attrs = [int(x) for x in masks[key]]
        masked_eval_attrs = [a for a in masked_attrs if a in cat_attr_ids or a in num_attr_ids]
        if len(masked_eval_attrs) == 0:
            continue

        preds = predict_single_row_for_attrs(model, r, masked_eval_attrs)

        for a in masked_eval_attrs:
            if a in cat_attr_ids:
                y_true = int(cat_full[r, a])
                if y_true == -1:
                    continue
                y_pred = int(preds[a])

                cat_y_true.append(y_true)
                cat_y_pred.append(y_pred)
                per_attr_cat_true[a].append(y_true)
                per_attr_cat_pred[a].append(y_pred)

            elif a in num_attr_ids:
                y_true = float(num_full[r, a])  # standardized => normalized MAE
                if y_true == -1:
                    continue
                y_pred = float(preds[a])

                err = abs(y_true - y_pred)
                num_abs_errors.append(err)
                per_attr_num_errors[a].append(err)

    weighted_f1 = f1_score(cat_y_true, cat_y_pred, average="weighted", zero_division=0) if len(cat_y_true) else np.nan
    cat_acc = accuracy_score(cat_y_true, cat_y_pred) if len(cat_y_true) else np.nan
    weighted_normalized_mae = float(np.mean(num_abs_errors)) if len(num_abs_errors) else np.nan

    if len(per_attr_num_errors):
        per_attr_mae = {a: float(np.mean(v)) for a, v in per_attr_num_errors.items()}
        per_attr_n = {a: len(v) for a, v in per_attr_num_errors.items()}
        weighted_attr_normalized_mae = sum(per_attr_mae[a] * per_attr_n[a] for a in per_attr_mae) / sum(per_attr_n.values())
    else:
        weighted_attr_normalized_mae = np.nan

    if len(per_attr_cat_true):
        per_attr_f1 = {
            a: f1_score(per_attr_cat_true[a], per_attr_cat_pred[a], average="weighted", zero_division=0)
            for a in per_attr_cat_true
        }
        per_attr_n = {a: len(per_attr_cat_true[a]) for a in per_attr_cat_true}
        weighted_attr_f1 = sum(per_attr_f1[a] * per_attr_n[a] for a in per_attr_f1) / sum(per_attr_n.values())
    else:
        weighted_attr_f1 = np.nan

    return {
        "scenario": payload["missing_scenario"],
        "missing_ratio": float(payload["missing_ratio"]),
        "mask_id": int(payload["mask_id"]),
        "weighted_f1": float(weighted_f1) if not np.isnan(weighted_f1) else np.nan,
        "weighted_attr_f1": float(weighted_attr_f1) if not np.isnan(weighted_attr_f1) else np.nan,
        "cat_accuracy": float(cat_acc) if not np.isnan(cat_acc) else np.nan,
        "weighted_normalized_mae": float(weighted_normalized_mae) if not np.isnan(weighted_normalized_mae) else np.nan,
        "weighted_attr_normalized_mae": float(weighted_attr_normalized_mae) if not np.isnan(weighted_attr_normalized_mae) else np.nan,
        "cat_eval_count": len(cat_y_true),
        "num_eval_count": len(num_abs_errors),
        "mask_file": mask_file_path,
    }

# ============================================================
# 16. RUN ALL EVALUATIONS
# ============================================================

all_results = []

print("\n===== EVALUATING SAVED MASKS =====")
for scenario in SCENARIOS:
    for ratio in MISSING_RATIOS:
        for mask_id in range(NUM_MASKS_PER_SCENARIO):
            mask_path = os.path.join(
                MASKS_DIR,
                scenario.lower(),
                ratio_to_folder_name(ratio),
                f"mask_{mask_id}.json"
            )
            if not os.path.exists(mask_path):
                continue

            res = evaluate_mask_file(model, mask_path)
            all_results.append(res)
            print(
                f"{scenario} ratio={ratio:.1f} mask={mask_id} | "
                f"weighted_f1={res['weighted_f1']:.4f} | "
                f"weighted_normalized_mae={res['weighted_normalized_mae']:.4f}"
            )

results_df = pd.DataFrame(all_results)
if len(results_df) == 0:
    raise RuntimeError("No results found. Check mask files.")

summary_df = (
    results_df
    .groupby(["scenario", "missing_ratio"], as_index=False)
    .agg(
        num_runs=("mask_id", "count"),
        weighted_f1_mean=("weighted_f1", "mean"),
        weighted_f1_std=("weighted_f1", "std"),
        weighted_attr_f1_mean=("weighted_attr_f1", "mean"),
        weighted_attr_f1_std=("weighted_attr_f1", "std"),
        weighted_normalized_mae_mean=("weighted_normalized_mae", "mean"),
        weighted_normalized_mae_std=("weighted_normalized_mae", "std"),
        weighted_attr_normalized_mae_mean=("weighted_attr_normalized_mae", "mean"),
        weighted_attr_normalized_mae_std=("weighted_attr_normalized_mae", "std"),
        cat_eval_count_mean=("cat_eval_count", "mean"),
        num_eval_count_mean=("num_eval_count", "mean"),
    )
    .sort_values(["scenario", "missing_ratio"])
    .reset_index(drop=True)
)

print("\n===== PER-MASK RESULTS =====")
print(results_df.head(20).to_string(index=False))

print("\n===== SUMMARY RESULTS =====")
print(summary_df.to_string(index=False))

results_path = os.path.join(OUT_DIR, "grimp_fasttext_init_per_mask_results.csv")
summary_path = os.path.join(OUT_DIR, "grimp_fasttext_init_summary_results.csv")

results_df.to_csv(results_path, index=False)
summary_df.to_csv(summary_path, index=False)

print("\nSaved:")
print(results_path)
print(summary_path)
print("\nDone.")